# 0.1 Import Libraries

In [ ]:
import re
from pathlib import Path

import pandas as pd

from bioimage_pipeline.data import find_tiff_files, load_tiff_image
from bioimage_pipeline.features import (
    extract_region_features,
    plot_area_vs_intensity,
    plot_feature_distribution,
    save_feature_summary,
    save_feature_table,
    summarize_feature_table,
)

# 0.2 Set Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / "data" / "raw" / "Fluo-C3DL-MDA231"
SEQUENCE_ID = "01"
MASK_DIR = PROJECT_ROOT / "reports" / "milestone_2" / "predicted_masks"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "milestone_3"

# 1.1 Load Predicted Segmentation Masks

In [ ]:
frame_pattern = re.compile(r"\d+")
mask_files = find_tiff_files(MASK_DIR, recursive=False)
mask_by_frame = {
    frame_pattern.search(path.stem).group(0).zfill(3): path
    for path in mask_files
    if frame_pattern.search(path.stem)
}
mask_by_frame

# 1.2 Load Matching Raw Images

In [ ]:
raw_files = find_tiff_files(DATASET_DIR / SEQUENCE_ID, recursive=False)
raw_by_frame = {
    frame_pattern.search(path.stem).group(0).zfill(3): path
    for path in raw_files
    if frame_pattern.search(path.stem)
}
matched_frames = sorted(set(mask_by_frame) & set(raw_by_frame))
matched_frames

# 2.1 Extract Region-Level Features

In [ ]:
frame_tables = []
for frame in matched_frames:
    mask_path = mask_by_frame[frame]
    raw_path = raw_by_frame[frame]
    label_image = load_tiff_image(mask_path)
    raw_image = load_tiff_image(raw_path)
    intensity_image = raw_image if raw_image.shape == label_image.shape else None
    frame_tables.append(
        extract_region_features(
            label_image,
            intensity_image=intensity_image,
            metadata={
                "dataset": DATASET_DIR.name,
                "sequence": SEQUENCE_ID,
                "frame": frame,
                "source_image_file": str(raw_path),
                "mask_file": str(mask_path),
                "method": "otsu",
            },
        )
    )

features = pd.concat(frame_tables, ignore_index=True, sort=False)
features.head()

# 2.2 Add Dataset and Frame Metadata

In [ ]:
features[["dataset", "sequence", "frame", "source_image_file", "mask_file", "method"]].head()

# 3.1 Create Feature Summary

In [ ]:
feature_summary = summarize_feature_table(features)
feature_summary

# 3.2 Visualize Feature Distributions

In [ ]:
FIGURES_DIR = OUTPUT_DIR / "figures"
plot_feature_distribution(features, "area", FIGURES_DIR / "cell_area_distribution.png")
if "mean_intensity" in features.columns:
    plot_feature_distribution(
        features,
        "mean_intensity",
        FIGURES_DIR / "mean_intensity_distribution.png",
    )
    plot_area_vs_intensity(features, FIGURES_DIR / "area_vs_intensity.png")

# 4.1 Export Feature Tables

In [ ]:
FEATURES_DIR = OUTPUT_DIR / "features"
feature_path = save_feature_table(features, FEATURES_DIR / "mda231_cell_features.csv")
summary_path = save_feature_summary(
    feature_summary,
    FEATURES_DIR / "mda231_feature_summary.csv",
)
feature_path, summary_path

# 5.1 Summary and Next Steps

Review feature distributions and segmentation-derived outliers before interpreting these exploratory measurements biologically.